# Hyperparameters

In [ ]:
N_FEATURES = 150
NUM_SAMPLES = 100000
NUM_CLUSTER = 150

K_NN = 7

# NUMBER_OF_DENSE_SIFT 

## SETTING

In [ ]:
DENSE = True
SOFT_ASSIGNMENT = True

## Import images

In [ ]:
from pathlib import Path

percorso_train = Path("dataset") / "train"
percorso_test = Path("dataset") / "test"

train_paths, train_label_ids, classi = read_split(percorso_train)
test_paths, test_label_ids, classi_test = read_split(percorso_test)

if classi != classi_test:
    raise ValueError(
        f"Le classi di train e test non corrispondono: {classi} != {classi_test}"
    )

train_labels = np.array([classi[label_id] for label_id in train_label_ids])
test_labels = np.array([classi[label_id] for label_id in test_label_ids])

print(f"Trovate {len(train_paths)} immagini di TRAINING.")
print(f"Trovate {len(test_paths)} immagini di TEST.")
print(f"Classi: {classi}")

if train_paths:
    prima_immagine = cv2.imread(str(train_paths[0]), cv2.IMREAD_GRAYSCALE)
    if prima_immagine is None:
        raise ValueError(f"Impossibile leggere l'immagine: {train_paths[0]}")

    plt.imshow(prima_immagine, cmap="gray")
    plt.title(f"Etichetta: {train_labels[0]}")
    plt.axis("off")
    plt.show()
else:
    print("ERRORE: non sono state trovate immagini in dataset/train.")

## SIFT DESCRIPTOR COMPUTATION

In [ ]:
sift = cv2.SIFT_create(nfeatures=N_FEATURES)

all_keypoints = []
all_descriptors = []
keypoint_counts = []
keypoint_labels = []


for image_path, label in zip(train_paths, train_labels):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Impossibile leggere l'immagine: {image_path}")
    if DENSE:
        # Generiamo i keypoint sulla griglia e calcoliamo i descrittori
        dense_keypoints = generate_dense_keypoints(image.shape, step_size=15, scales=[10, 20])
        keypoints, descriptors = sift.compute(image, dense_keypoints)
    else:
        keypoints, descriptors = sift.detectAndCompute(image, None) 


    keypoint_counts.append(len(keypoints))
    keypoint_labels.append(label)

    if descriptors is not None:
        all_descriptors.append(descriptors)
        all_keypoints.append(keypoints)

keypoint_counts = np.array(keypoint_counts, dtype=np.int32)
keypoint_labels = np.array(keypoint_labels)


print(f"Immagini analizzate: {len(keypoint_counts)}")
print(f"Keypoint totali: {keypoint_counts.sum()}")






## Samplig of *N_FEATURES* features

In [ ]:
# 1. Stack all descriptor arrays vertically into a single massive matrix
stacked_descriptors = np.vstack(all_descriptors)

total_descriptors = stacked_descriptors.shape[0]
print(f"Total descriptors extracted: {total_descriptors}")

# 2. Set the target number of samples (10K to 100K as per assignment)
num_samples = NUM_SAMPLES

# 3. Perform random sampling
if total_descriptors > num_samples:
    print(f"Sampling {num_samples} descriptors randomly...")

    # Generate random row indices without replacement
    random_indices = np.random.choice(total_descriptors, num_samples, replace=False)

    # Filter the original matrix using the random indices
    sampled_descriptors = stacked_descriptors[random_indices, :]

else:
    print("Under the maximum limit, keeping all descriptors.")
    sampled_descriptors = stacked_descriptors

# Final verification (should output (100000, 128))
print(f"Ready for K-Means! Final array shape: {sampled_descriptors.shape}")

## Visual volabory creation

In [ ]:
# 1. Definiamo il numero di cluster (le nostre visual words)
num_clusters = NUM_CLUSTER

print(f"Inizio addestramento K-Means con k={num_clusters} (potrebbe volerci qualche minuto)...")
start_time = time.time()

# 2. Inizializziamo l'algoritmo K-Means
# random_state=42 garantisce che se lo rieseguite domani, vi darà gli stessi identici cluster
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)

# 3. Addestriamo il modello sui nostri dati campionati
kmeans.fit(sampled_descriptors)

end_time = time.time()
print(f"K-Means completato in {(end_time - start_time) / 60:.2f} minuti.")

# 4. Estraiamo i centroidi (il nostro vero e proprio vocabolario)
# Sarà una matrice di forma (num_cluster, 128)
visual_vocabulary = kmeans.cluster_centers_
print(f"Forma del vocabolario visivo: {visual_vocabulary.shape}")

## Histogram creation

### Training Histograms

In [ ]:
def compute_normalized_histogram(image_path, sift_detector, kmeans_model, soft_assignment):
    """
    Estrae i descrittori da un'immagine, li mappa sul vocabolario
    e restituisce l'istogramma normalizzato.
    """
    
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        # Se per qualche motivo l'immagine non si apre
        return None
    
    # 1. Estrazione SIFT
    _, descriptors = sift_detector.detectAndCompute(image, None)

    num_clusters = kmeans_model.n_clusters

    # Gestione sicurezza: se l'immagine è troppo piatta e non trova keypoints
    if descriptors is None or len(descriptors) == 0:
        return np.zeros(num_clusters)

    # 2. Assegnazione a quale cluster appartiene ciascun descrittore
    # predict() restituisce un array di interi (es. [12, 45, 3, 12, ...])
    visual_words = kmeans_model.predict(descriptors)

    # 3. Creazione dell'istogramma grezzo (conteggi per ogni cluster)
    # np.histogram conta quanti elementi finiscono in ciascun "bin" (da 0 a num_clusters)
    histogram, _ = np.histogram(visual_words, bins=num_clusters, range=(0, num_clusters))

    # 4. Normalizzazione (la somma totale deve fare 1.0)
    total_descriptors = np.sum(histogram)
    if total_descriptors > 0:
        normalized_histogram = histogram / total_descriptors
    else:
        normalized_histogram = histogram

    return normalized_histogram

# --- APPLICHIAMO A TUTTO IL TRAINING SET ---
print("Generazione degli istogrammi per il training set...")

X_train = []
y_train = []

for path, label in zip(train_paths, train_labels):
    hist = compute_normalized_histogram(path, sift, kmeans)
    if hist is not None:
        X_train.append(hist)
        y_train.append(label)

# Convertiamo in matrici NumPy definitive pronte per il Machine Learning
X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"Fatto! Matrice X_train creata con forma: {X_train.shape}")
print(f"Array y_train delle etichette creato con forma: {y_train.shape}")

### Test Histograms

In [ ]:
print("Generazione degli istogrammi per il TEST set...")

X_test = []
y_test = []

for path, label in zip(test_paths, test_labels):
    # Usiamo lo stesso SIFT e lo stesso vocabolario (kmeans) addestrato sul training!
    hist = compute_normalized_histogram(path, sift, kmeans)
    if hist is not None:
        X_test.append(hist)
        y_test.append(label)

X_test = np.array(X_test)
y_test = np.array(y_test)

print(f"Fatto! Matrice X_test creata con forma: {X_test.shape}")